In [2]:
import pandas as pd
import numpy as np

In [3]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [4]:
POSBal = pd.read_csv('data/Internal/POS_CASH_balance.csv')

In [5]:
POSBal.sort_values(by=['SK_ID_CURR','SK_ID_PREV','MONTHS_BALANCE'], inplace=True, ignore_index=True)

In [6]:
POSBal = POSBal.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [7]:
#check for installment prepayment

multiple installment counts

In [8]:
def get_active_count_last_year(x, df):
    """
    Count the number of active installments in the last year.
    """
    # Get the slice of the DataFrame relevant to the group x
    group_df = df.loc[x.index]
    
    # Create a boolean mask for the 'last year' condition
    last_year_mask = group_df['MONTHS_BALANCE'] > -12
    
    # Create a boolean mask for the 'Active' status condition
    active_mask = group_df['NAME_CONTRACT_STATUS'] == 'Active'
    
    return (last_year_mask & active_mask).sum()

In [9]:
POSBal_Silver = POSBal.groupby(['SK_ID_CURR','SK_ID_PREV']).agg(
  #  POS_AVG_DURATION = ('MONTHS_BALANCE', 'min'),
  #  POS_FIRST_CREDIT = ('MONTHS_BALANCE', 'min'),
    POS_ACTUAL_DURATN = ('MONTHS_BALANCE', 'size'),
    POS_EXPECTED_DURATN =  ('CNT_INSTALMENT', 'max'),
    POS_INSTALMENT_CNT_FUTURE = ('CNT_INSTALMENT_FUTURE', 'min'),
    POS_ACTIVE_CNT_LAST_YEAR = ('MONTHS_BALANCE', lambda x: get_active_count_last_year(x, POSBal)),
    POS_MAX_DPD = ('SK_DPD_DEF', 'max'),
    POS_DAYS_IN_DPD = ('SK_DPD_DEF', lambda x: sum(x > 0)),
    POS_LAST_DPD_STATUS = ('SK_DPD_DEF', 'last'), # last DPD status of Previous ID
    POST_LAST_DPD_DAYS = ('MONTHS_BALANCE', 'last') # how many days ago was POS_LAST_DPD_STATUS
).reset_index()

Prepayment Flag is actual duration < originalexpected duration

In [10]:
POSBal_Silver['PREPAY_COUNT'] = np.where(POSBal_Silver['POS_ACTUAL_DURATN'] < POSBal_Silver['POS_EXPECTED_DURATN'], 1, 0)

In [11]:
POSBal_Silver['DAYS_IN_DPD_TO_TOT_DURATION_RATIO'] = POSBal_Silver['POS_DAYS_IN_DPD'] / POSBal_Silver['POS_ACTUAL_DURATN']

In [12]:
POSBal_Silver.sort_values(by=['SK_ID_CURR','POST_LAST_DPD_DAYS'], inplace=True, ignore_index=True)

In [13]:
POSBal_Gold = POSBal_Silver.groupby(['SK_ID_CURR']).agg(
    POS_PREV_ID_COUNT = ('SK_ID_PREV', lambda x: x.nunique()),

    POS_AVG_ACTUAL_DURATN = ('POS_ACTUAL_DURATN', 'mean'),
    POS_MAX_ACTUAL_DURATN = ('POS_ACTUAL_DURATN', 'max'),
    POS_MEDIAN_ACTUAL_DURATN = ('POS_ACTUAL_DURATN', 'median'),

    POS_AVG_INSTALMENT_CNT_FUTURE = ('POS_INSTALMENT_CNT_FUTURE', 'mean'),
    POS_MAX_INSTALMENT_CNT_FUTURE = ('POS_INSTALMENT_CNT_FUTURE', 'max'),
    POS_MEDIAN_INSTALMENT_CNT_FUTURE = ('POS_INSTALMENT_CNT_FUTURE', 'median'),
    
    POS_AVG_ACTIVE_CNT_LAST_YEAR = ('PREPAY_COUNT', 'mean'),
    POS_MAX_ACTIVE_CNT_LAST_YEAR = ('PREPAY_COUNT', 'max'),
    POS_MEDIAN_ACTIVE_CNT_LAST_YEAR = ('PREPAY_COUNT', 'median'),

    POS_MAX_MAX_DPD = ('POS_MAX_DPD', 'max'),    

    POS_AVG_DAYS_IN_DPD = ('POS_DAYS_IN_DPD', 'mean'),
    POS_MAX_DAYS_IN_DPD = ('POS_DAYS_IN_DPD', 'max'),
    POS_MEDIAN_DAYS_IN_DPD = ('POS_DAYS_IN_DPD', 'median'),
    POS_AVG_DAYS_IN_DPD_TO_TOT_DURATION_RATIO = ('DAYS_IN_DPD_TO_TOT_DURATION_RATIO', 'mean'),

    POS_LAST_DPD_STATUS = ('POS_LAST_DPD_STATUS', 'last')
    
).reset_index()

In [14]:
POSBal_Gold.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\posdata_processed.parquet")